In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.review_analytics")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.review_analytics.raw_files")
print("Schema and volume created")

Schema and volume created


In [0]:
from functools import reduce

VOL = "/Volumes/workspace/review_analytics/raw_files"
DB = "workspace.review_analytics"

def save_delta(df, name):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(f"{DB}.{name}"))
    print(f"{name}: {spark.table(f'{DB}.{name}').count():,} rows")

yearly = []
for year in [2023, 2024, 2025, 2026]:
    df = spark.read.parquet(f"{VOL}/reviews_{year}.parquet")
    save_delta(df, f"reviews_{year}")
    yearly.append(spark.table(f"{DB}.reviews_{year}"))

save_delta(reduce(lambda a, b: a.unionByName(b), yearly), "reviews_all")
save_delta(spark.read.parquet(f"{VOL}/products.parquet"), "products")

reviews_2023: 220,000 rows
reviews_2024: 260,000 rows
reviews_2025: 310,000 rows
reviews_2026: 350,000 rows
reviews_all: 1,140,000 rows
products: 543,748 rows


In [0]:
from pyspark.sql import functions as F

reviews_all = spark.table("workspace.review_analytics.reviews_all")

display(reviews_all
        .groupBy("year", "category")
        .agg(F.count("*").alias("reviews"),
             F.round(F.avg("rating"), 2).alias("avg_rating"))
        .orderBy("year", "category"))

year,category,reviews,avg_rating
2023,Beauty and Personal Care,37400,4.13
2023,Books,26400,4.45
2023,Electronics,48400,4.24
2023,Health and Household,35200,4.22
2023,Home and Kitchen,44000,4.23
2023,Sports and Outdoors,28600,4.19
2024,Beauty and Personal Care,46800,4.0
2024,Books,26000,4.43
2024,Electronics,52000,4.15
2024,Health and Household,44200,4.11
